In [0]:
# Step 1 - Read Bronze Table
from pyspark.sql.functions import *

spark.sql("USE CATALOG Silver_Catalog")
spark.sql("USE SCHEMA Silver_SCH")

weather_df = spark.table(
    "Bronze_Catalog.Bronze_SCH.Bronze_Weather"
)

In [0]:
display(weather_df.limit(10))
print(weather_df.count())

In [0]:
# Step 2 - Verify Data
weather_df.printSchema()


In [0]:
# Step 3 - Check Duplicate Records

# Business Key = household_id + timestamp


print("Total Records :", weather_df.count())

print(
    "Distinct Records :",
    weather_df.select(
        "household_id",
        "timestamp"
    ).distinct().count()
)

In [0]:
# Step 4 - Remove Duplicates
weather_df = weather_df.dropDuplicates(
    [
        "household_id",
        "timestamp"
    ]
)

# Verify:

print(weather_df.count())

In [0]:
# Step 5 - Null Count
null_df = weather_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in weather_df.columns
])

display(null_df)

In [0]:
from pyspark.sql.functions import expr

weather_df = weather_df \
.withColumn("temperature_celsius", expr("try_cast(temperature_celsius as double)")) \
.withColumn("humidity_percent", expr("try_cast(humidity_percent as double)")) \
.withColumn("wind_speed_kmh", expr("try_cast(wind_speed_kmh as double)")) \
.withColumn("rainfall_mm", expr("try_cast(rainfall_mm as double)")) \
.withColumn("pressure_hpa", expr("try_cast(pressure_hpa as double)")) \
.withColumn("solar_radiation", expr("try_cast(solar_radiation as double)")) \
.withColumn("dew_point", expr("try_cast(dew_point as double)")) \
.withColumn("uv_index", expr("try_cast(uv_index as double)")) \
.withColumn("visibility_km", expr("try_cast(visibility_km as double)")) \
.withColumn("cloud_cover_percent", expr("try_cast(cloud_cover_percent as double)"))

In [0]:
# Step 7 - Handle Null Values

from pyspark.sql.functions import col

weather_df = weather_df.fillna({
    "temperature_celsius": 0,
    "humidity_percent": 0,
    "wind_speed_kmh": 0,
    "rainfall_mm": 0,
    "pressure_hpa": 0,
    "solar_radiation": 0,
    "dew_point": 0,
    "uv_index": 0,
    "visibility_km": 0,
    "cloud_cover_percent": 0
})

In [0]:
from pyspark.sql.functions import col

weather_df = weather_df.filter(
    (col("temperature_celsius") >= 0) &
    (col("humidity_percent") >= 0) &
    (col("wind_speed_kmh") >= 0) &
    (col("rainfall_mm") >= 0) &
    (col("pressure_hpa") >= 0) &
    (col("solar_radiation") >= 0) &
    (col("dew_point") >= 0) &
    (col("uv_index") >= 0) &
    (col("visibility_km") >= 0) &
    (col("cloud_cover_percent") >= 0)
)

print("Valid Records :", weather_df.count())

In [0]:
from pyspark.sql.functions import sum, col

weather_df.select(
    [
        sum(col(c).isNull().cast("int")).alias(c)
        for c in [
            "temperature_celsius",
            "humidity_percent",
            "wind_speed_kmh",
            "rainfall_mm",
            "pressure_hpa",
            "solar_radiation",
            "dew_point",
            "uv_index",
            "visibility_km",
            "cloud_cover_percent"
        ]
    ]
).show()

In [0]:
from pyspark.sql.functions import current_timestamp

records_loaded = weather_df.count()

try:

    spark.sql(f"""
    INSERT INTO Silver_Catalog.Silver_SCH.Audit_Log
    VALUES(
        'Silver_Weather',
        'Weather Silver Pipeline',
        current_timestamp(),
        {records_loaded},
        'SUCCESS',
        NULL
    )
    """)

    print(f" Silver_Weather loaded successfully. Records Loaded: {records_loaded}")

   

except Exception as e:

    error_message = str(e).replace("'", "''")

    spark.sql(f"""
    INSERT INTO Silver_Catalog.Silver_SCH.Audit_Log
    VALUES(
        'Silver_Weather',
        'Weather Silver Pipeline',
        current_timestamp(),
        0,
        'FAILED',
        '{error_message}'
    )
    """)
    print(f"❌ Silver_Weather failed : {error_message}")


    raise

In [0]:
# 
# Step 9 - Create Surrogate Key
from pyspark.sql.functions import monotonically_increasing_id

weather_df = weather_df.withColumn(
    "weather_sk",
    monotonically_increasing_id()
)

weather_df.select(
    "weather_sk",
    "household_id"
).show(10, False)

In [0]:
# Step 10 - Add SCD Type 2 Columns
from pyspark.sql.functions import current_timestamp, lit

weather_df = weather_df \
    .withColumn("effective_from", current_timestamp()) \
    .withColumn("effective_to", lit(None).cast("timestamp")) \
    .withColumn("is_current", lit(True))

display(weather_df)

In [0]:
# Step 11 - Save Silver Table
weather_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable("Silver_Catalog.Silver_SCH.Silver_Weather")

In [0]:
from pyspark.sql.functions import current_timestamp

records_loaded = weather_df.count()

try:

    spark.sql(f"""
    INSERT INTO Silver_Catalog.Silver_SCH.Audit_Log
    VALUES(
        'Silver_Weather',
        'Weather Silver Pipeline',
        current_timestamp(),
        {records_loaded},
        'SUCCESS',
        NULL
    )
    """)

    print("Silver Weather Loaded Successfully")

except Exception as e:

    error_message = str(e).replace("'", "''")

    spark.sql(f"""
    INSERT INTO Silver_Catalog.Silver_SCH.Audit_Log
    VALUES(
        'Silver_Weather',
        'Weather Silver Pipeline',
        current_timestamp(),
        0,
        'FAILED',
        '{error_message}'
    )
    """)

    raise